# AGROBOT — entrenamiento en Kaggle
Activa **GPU** (Settings → Accelerator) e **Internet ON**. El repo trae el dataset sintético (`percepcion_sintetica/datos/*.npz`).
Si el repo es privado, añade un token en *Add-ons → Secrets* con nombre `GH_TOKEN`.

In [ ]:
import os, subprocess
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret('GH_TOKEN')
    url = 'https://' + tok + '@github.com/DAvolmx/AGROBOT_V1.git'
except Exception:
    url = 'https://github.com/DAvolmx/AGROBOT_V1.git'
if not os.path.exists('AGROBOT_V1'):
    subprocess.run(['git','clone','--depth','1','--branch','mejoras-auditoria-20261007',url], check=True)
%cd AGROBOT_V1/agrobot_system_2/simulador_pasivo/percepcion_sintetica
!ls datos/train | wc -l
!pip install -q ultralytics segmentation-models-pytorch timm

## 1. Segmentación de navegación (UNet, 5 clases)

In [ ]:
!python entrenar_seg.py --epocas 24 --batch 32

## 2. Detector de objetos de interior (YOLO11n: maceta/obstáculo/borde)

In [ ]:
!python entrenar_yolo.py --tarea obj --epocas 40 --batch 32

## 3. Mosca blanca (opcional): descarga datos abiertos y entrena
Es un modelo PARCIAL preentrenado con yuca; no está validado en col/acelga.

In [ ]:
!python descargar_plagas.py
!python entrenar_whitefly.py --epocas 40 --batch 16 --workers 2

## 4. Recoger resultados
Descarga `/kaggle/working/resultados.zip` desde *Output* y cópialo a la Jetson (ver `agrobot_system_2/kaggle/LEEME.md`).

In [ ]:
import glob, shutil
os_ = glob.glob('modelos/**/*.pt', recursive=True) + glob.glob('modelos/*.json')
os.makedirs('/kaggle/working/res', exist_ok=True)
for f in os_:
    d = '/kaggle/working/res/' + f.replace('/', '__'); shutil.copy(f, d)
shutil.make_archive('/kaggle/working/resultados', 'zip', '/kaggle/working/res')
print(len(os_), 'archivos')